# easy-capture 앱 코드 검증 노트북

## 목적

이 노트북은 **앱 패키지(`src/easy_capture`)를 직접 import해** 추적→크롭→GIF/MP4 파이프라인을 검증한다.

### 기존 PoC 노트북(`easy_capture_gpu_poc.ipynb`)과의 차이

| 항목 | PoC 노트북 | **이 노트북(앱 검증)** |
|------|-----------|------------------------|
| 크롭 방식 | 셀 9에서 `ffmpeg crop=320:320:0:0` (좌상단 고정 더미) | `VideoCaptureUseCase.compute_boxes()` — 추적 centroid 기반 추적 크롭 |
| 코드 경로 | 인라인 스크립트(앱 미사용) | **앱 패키지 공개 API 그대로 사용** |
| 검증 목적 | H1·H2·H4 개별 검증 | **AC-01(추적 유지율), AC-06(GPU 속도) + 크롭 정합성** |

## 사용법

1. 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU(T4)** 로 설정한다.
2. 셀을 **위에서 아래로** 순서대로 실행한다.
3. **짧은 MV/직캠 클립(≤10초 권장)** 을 업로드한다.
4. 검출된 후보 그림을 보고 `TARGET_IDX` 변수를 지정한 뒤 나머지 셀을 실행한다.
5. `clip_crop.gif`/`clip_crop.mp4`에서 **크롭이 좌상단 고정이 아니라 피사체를 따라가는지** 확인한다.

## 검증 항목

- **AC-01** 추적 유지율 ≥ 80%
- **AC-06** GPU 처리 속도 ≥ 10 fps
- **크롭 정합성** — clip_crop 결과물이 좌상단 배경이 아닌 피사체 중심인지
- **마스크 해상도 정합** — `_original_hw` 수정 후 centroid 좌표계 오류 여부

In [ ]:
# ───────────────────────────────────────────────
# 셀 1: 앱 패키지 + 의존성 설치 + GPU 확인
# ───────────────────────────────────────────────
# WHY: src 레이아웃 + editable(-e) 설치는 Colab 커널이 sys.path 를 못 잡는 일이 잦다
#      (ModuleNotFoundError: easy_capture). 또 pyproject 의존성에 PySide6(무겁고
#      헤드리스에 불필요)가 있어 -e 설치가 통째로 실패할 수 있다.
#      → 앱 패키지는 src 를 sys.path 에 직접 추가하고(editable path 문제 회피),
#        런타임에 실제 필요한 의존성만 명시 설치한다. (앱 core/app 은 torch/av/
#        transformers/scenedetect 를 함수 내부에서 지연 import 하므로 import 자체엔
#        numpy 만 있으면 된다 — 무거운 패키지 설치 실패가 import 를 막지 않는다.)
!git clone -b feature/video/crop-tuning \
    https://github.com/JunOnJuly-Project/easy-capture.git easy-capture-verify 2>&1 | tail -3

# 런타임에 실제 쓰는 의존성만 설치 (PySide6 제외). torch 는 Colab 기본 제공.
%pip install -q -U "transformers>=4.55" accelerate "scenedetect[opencv]" av imageio imageio-ffmpeg

# 앱 패키지: src 를 sys.path 에 직접 추가 → editable 설치 path 문제 회피
import sys, importlib
SRC = "/content/easy-capture-verify/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

# import 검증 — 여기서 통과하면 이후 셀의 from easy_capture ... 가 모두 동작
import easy_capture
print("easy_capture OK:", easy_capture.__file__)

import torch
print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("경고: GPU 런타임이 아닙니다. 런타임 유형을 GPU 로 바꾸고 다시 실행하세요.")

In [ ]:
# ───────────────────────────────────────────────
# 셀 2: 클립 업로드 + 구간 설정 + 메타 확인
# ───────────────────────────────────────────────
from google.colab import files
from easy_capture.infra.video_io import open_source, FrameSpan

# 처리할 최대 길이(초) — 너무 길면 GPU 메모리 부족 위험
MAX_SECONDS = 8

print("짧은 MV/직캠 클립(≤10초 권장)을 업로드하세요.")
uploaded = files.upload()
VIDEO_PATH = next(iter(uploaded))  # 업로드된 첫 번째 파일 경로

# open_source: 확장자로 이미지/영상을 판별 후 FrameSource 구현체 반환
source = open_source(VIDEO_PATH)
meta = source.probe()  # FrameMeta(width, height, is_video, fps)

print(f"소스: {VIDEO_PATH}")
print(f"해상도: {meta.width}x{meta.height} | fps: {meta.fps:.2f} | 동영상: {meta.is_video}")

# 구간 설정: 0프레임부터 MAX_SECONDS * fps 프레임까지
# WHY: FrameSpan(start, end, step) — end는 exclusive
FPS = meta.fps if meta.fps else 30.0
END_FRAME = int(FPS * MAX_SECONDS)
SPAN = FrameSpan(start=0, end=END_FRAME, step=1)
print(f"처리 구간: 프레임 0 ~ {END_FRAME} ({MAX_SECONDS}초 @ {FPS:.1f}fps)")

In [ ]:
# ───────────────────────────────────────────────
# 셀 3: 구간 프레임 추출
# ───────────────────────────────────────────────
# VideoCaptureUseCase는 나중에 조립하지만
# 프레임 추출만 먼저 수행해 메모리/소요 시간을 미리 파악한다.
# WHY: source.read_frames(span)가 앱 공개 API — PyAV 내부 구현에 의존하지 않는다.

print("프레임 추출 중...")
frames = source.read_frames(SPAN)

H, W = frames[0].shape[:2]
print(f"추출 완료: {len(frames)}프레임, {W}x{H}")

In [ ]:
# ───────────────────────────────────────────────
# 셀 4: 첫 프레임 후보 검출 + 표시
# ───────────────────────────────────────────────
# GroundingDinoBackend: 지연 로드 — 이 셀에서 처음 detect() 호출 시 모델 로드
import cv2
import numpy as np
from PIL import Image
from IPython.display import display

from easy_capture.infra.grounding_dino_backend import GroundingDinoBackend

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# GroundingDinoBackend 생성자: repo, device, box_threshold, text_threshold
# WHY: 기본값(grounding-dino-base, 0.30, 0.25)을 그대로 사용
detector = GroundingDinoBackend(device=DEVICE)

# 검출 프롬프트 — 사람 이외 피사체는 변경(예: "car." / "dog.")
DETECT_PROMPT = "person."

print("Grounding DINO 모델 로드 + 첫 프레임 검출 중...")
# detect(frame, prompt) → Detection 리스트(score 내림차순)
candidates = detector.detect(frames[0], DETECT_PROMPT)

# 후보 박스를 첫 프레임에 그려서 표시
vis = frames[0].copy()
for i, det in enumerate(candidates):
    x1, y1, x2, y2 = [int(v) for v in det.box]  # Detection.box: (x1,y1,x2,y2)
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(
        vis, f"{i}:{det.score:.2f}",
        (x1, max(16, y1 - 6)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2,
    )

display(Image.fromarray(vis))
print(f"검출 후보 {len(candidates)}개. 다음 셀에서 TARGET_IDX를 지정하세요.")

In [ ]:
# ───────────────────────────────────────────────
# 셀 5: 추적 대상 선택
# ───────────────────────────────────────────────
# 위 그림에서 추적할 후보의 인덱스를 지정한다.
# WHY: 인덱스 0 = 가장 높은 confidence 후보(score 내림차순 정렬)
TARGET_IDX = 0  # 원하는 후보 인덱스로 변경

if TARGET_IDX >= len(candidates):
    raise IndexError(
        f"TARGET_IDX={TARGET_IDX}가 후보 수({len(candidates)})를 초과합니다."
    )

target_box = candidates[TARGET_IDX].box  # (x1, y1, x2, y2)
# 박스 중심을 클릭 포인트로 변환 — VideoCaptureUseCase.track()의 point 인자
# WHY: SAM2 add_click(point=(x,y)) 인자 형식 (정수)
CLICK_POINT = (
    int((target_box[0] + target_box[2]) / 2),
    int((target_box[1] + target_box[3]) / 2),
)
print(f"선택 후보 {TARGET_IDX}: box={target_box}, click_point={CLICK_POINT}")

In [ ]:
# ───────────────────────────────────────────────
# 셀 6: 백엔드 + UseCase 조립
# ───────────────────────────────────────────────
from easy_capture.infra.sam2_video_backend import Sam2VideoBackend
from easy_capture.app.video_capture import VideoCaptureUseCase

# SAM2 video 백엔드: repo(transformers Hub ID), device
# WHY: sam2.1-hiera-tiny가 Colab T4 VRAM에 여유 있게 올라감(PoC 검증 노트북과 동일)
SAM2_REPO = "facebook/sam2.1-hiera-tiny"
backend = Sam2VideoBackend(repo=SAM2_REPO, device=DEVICE)

# VideoCaptureUseCase(source, backend, detector=None)
# detector를 주입하면 샷 경계 재추적 모드(컷 감지 + 재매칭)
# WHY: 멀티샷 재매칭 검증을 위해 detector 주입
usecase = VideoCaptureUseCase(source=source, backend=backend, detector=detector)
print("UseCase 조립 완료 (모델은 track() 첫 호출 시 지연 로드됨)")

In [ ]:
# ───────────────────────────────────────────────
# 셀 7: 컷 감지
# ───────────────────────────────────────────────
# detect_cut_frames: CPU에서 동작 — 모델 로드 없음, 빠름
# 반환: 구간 내 상대 프레임 인덱스 리스트(절대 프레임 번호 아님)
from easy_capture.infra.shot_detect import detect_cut_frames

# threshold=27.0 기본값 — 낮을수록 민감(과검출), 높을수록 둔감(미검출)
# WHY: ContentDetector 기본값 27.0을 그대로 사용 (PoC h2와 동일)
cut_frames = detect_cut_frames(
    VIDEO_PATH,
    start_frame=SPAN.start,
    end_frame=SPAN.end,
)
print(f"감지된 컷 프레임(구간 내 상대 인덱스): {cut_frames}")

In [ ]:
# ───────────────────────────────────────────────
# 셀 8: 추적 실행 (무거움)
# ───────────────────────────────────────────────
# VideoCaptureUseCase.track(frames, point, cut_frames)
#   - cut_frames + detector 주입 → 다중 샷 경로(샷 경계 재추적)
#   - 반환: TrackResult(masks, centroids, needs_correction, cut_frames)
# WHY: SAM2 모델 지연 로드는 이 셀 첫 실행 시 수행(수~수십 초 소요)
import time

print("SAM2 모델 로드 + 추적 시작... (첫 실행 시 모델 다운로드 수 분 소요)")
t_start = time.time()

# track(frames, point, cut_frames=cut_frames)
# point = (x, y) 정수 튜플
track_result = usecase.track(frames, CLICK_POINT, cut_frames=cut_frames)

elapsed = time.time() - t_start
n_frames = len(frames)

# TrackResult.centroids: centroid가 None이면 해당 프레임 추적 실패(occlusion)
n_tracked = sum(1 for c in track_result.centroids if c is not None)
retention_pct = 100.0 * n_tracked / n_frames
gpu_fps = n_frames / elapsed

# needs_correction: True이면 컷 재매칭 미달 구간
n_corrected = sum(track_result.needs_correction)

print("\n=== 추적 결과 ===")
print(f"총 프레임: {n_frames}")
print(f"추적 유지율(AC-01): {retention_pct:.1f}%  ({n_tracked}/{n_frames}프레임) [목표: ≥80%]")
print(f"GPU 처리 속도(AC-06): {gpu_fps:.1f} fps  (총 {elapsed:.1f}s) [목표: ≥10fps]")
print(f"needs_correction 구간: {n_corrected}프레임")
print(f"감지 컷(TrackResult.cut_frames): {track_result.cut_frames}")

In [ ]:
# ───────────────────────────────────────────────
# 셀 9: 크롭 박스 산출 (핵심 검증)
# ───────────────────────────────────────────────
# VideoCaptureUseCase.compute_boxes(result, params, frame_size)
#   - result: TrackResult (셀 8 결과)
#   - params: VideoCropParams(box_size, aspect, smooth_window, subject_padding)
#   - frame_size: (W, H)
# 반환: list[(x1, y1, x2, y2)] — 전 프레임 동일 크기 보장
#
# WHY(크롭 정책): 중심은 마스크 bbox 중심(무게중심 centroid 대신 → 자세 변화 흔들림↓),
#      크기는 구간 내 최대 피사체 bbox × subject_padding 으로 한 번 자동 고정한다.
#      → 피사체가 항상 박스 안에 담겨 잘리지 않고(고정 320 한계 제거),
#        크기가 고정이라 줌 흔들림이 없다(사용자 피드백 반영).
from easy_capture.app.video_capture import VideoCropParams

# box_size 는 이제 '최소 크기 하한' — 실제 크기는 피사체 bbox × subject_padding 으로 자동.
# 조절 포인트:
#   - subject_padding ↑ (예 1.6) : 피사체 주변 여백 ↑
#   - smooth_window ↑ (예 9, 13)  : 위치 흔들림 ↓ (이동평균 윈도)
#   - aspect: "1:1" / "9:16" / "16:9" / None — 잘림 없이 짧은 변을 늘려 비율 맞춤
MIN_W = 320
MIN_H = 320
ASPECT = "1:1"        # None = 자유 비율(피사체 bbox 그대로)
SMOOTH_WINDOW = 9     # 흔들림 완화 (기본 5 → 9)
SUBJECT_PADDING = 1.4 # 피사체 여백 배수

crop_params = VideoCropParams(
    box_size=(MIN_W, MIN_H),
    aspect=ASPECT,
    smooth_window=SMOOTH_WINDOW,
    subject_padding=SUBJECT_PADDING,
)

# frame_size = (W, H) 순서
boxes = usecase.compute_boxes(
    result=track_result,
    params=crop_params,
    frame_size=(W, H),
)

print(f"크롭 박스 산출 완료: {len(boxes)}개")
# 박스 크기 — 피사체를 담도록 box_size(320)보다 커질 수 있다(자동 크기)
sizes = {(b[2] - b[0], b[3] - b[1]) for b in boxes}
print(f"박스 크기: {sizes}  (1종이어야 정상 / 320 이상이면 피사체에 맞춰 확대된 것)")
# 첫 5개 박스 위치 — 프레임마다 피사체를 따라 이동
print("첫 5개 박스 (x1,y1,x2,y2):", boxes[:5])

In [ ]:
# ───────────────────────────────────────────────
# 셀 9.5: 슬로우모션 구간 (선택) — Story 6
# ───────────────────────────────────────────────
# 특정 프레임 구간을 슬로우(factor<1, 느리게)/패스트(factor>1, 빠르게)로 만든다. 다중 구간 가능.
# 인덱스는 추출한 frames 기준 상대 인덱스 [0, len(frames)). 데스크톱 UI와 동일한
# core 함수(build_playback_schedule 등)를 사용해 재현성을 보장한다.
from easy_capture.core.timing.timeremap import (
    SpeedSegment,
    build_playback_schedule,
    clamp_durations_for_gif,
    estimate_output_frame_count,
)

# 예: 프레임 30~60 구간을 0.5x(2배 느리게). 비우면 () = 등속(슬로우모션 없음).
# 배속 프리셋: 0.25 / 0.33 / 0.5(느리게) · 1.0(등속) · 2.0(빠르게). 다중: 여러 SpeedSegment 나열.
SEGMENTS = (
    SpeedSegment(start=30, end=60, factor=0.5),
)
# SEGMENTS = ()  # 슬로우모션 끄기

# 스케줄 요약 미리보기 (Task 6-2)
if SEGMENTS:
    _n = len(frames)
    _schedule = build_playback_schedule(_n, list(SEGMENTS), FPS)
    _, _clamped = clamp_durations_for_gif(_schedule)
    _est_mp4 = estimate_output_frame_count(_n, SEGMENTS, FPS)
    print(f"구간 {len(SEGMENTS)}개 · 원본 {_n}프레임")
    print(f"MP4 예상 출력 프레임: {_est_mp4} (원본 대비 {_est_mp4 / _n:.2f}배)")
    if _clamped:
        print(
            f"⚠ GIF 패스트 {len(_clamped)}프레임이 10ms 미만 → 20ms 클램프(역전 방지). "
            "fps를 낮추거나 배속을 줄이세요."
        )
    if _est_mp4 > _n * 3:
        print(f"⚠ 출력 프레임 폭증({_est_mp4}) — 인코딩 시간·용량 주의.")
else:
    print("SEGMENTS=() — 슬로우모션 없음(등속 재생).")

In [ ]:
# ───────────────────────────────────────────────
# 셀 10: 크롭 GIF + MP4 export (슬로우모션 segments 적용)
# ───────────────────────────────────────────────
# usecase.export(frames, boxes, target, result=track_result)
#   target = (출력경로, VideoExportConfig)
#   result를 넘겨야 valid_flags가 centroids에서 정확히 도출됨(occlusion 프레임 처리)
#   segments(셀 9.5)를 config에 넣으면 구간별 슬로우/패스트 적용(데스크톱 UI와 동일 경로)
from easy_capture.core.export.video_export import VideoExportConfig
from easy_capture.core.tracking.gap_policy import GapPolicy

GIF_PATH = "clip_crop.gif"
MP4_PATH = "clip_crop.mp4"

# ── GIF 재생속도 ──────────────────────────────
# 기본은 원본 영상과 동일(FPS). 더 느리게/빠르게 하려면 직접 값 지정.
#   예) 느리게: GIF_FPS = 8.0   /   빠르게: GIF_FPS = 30.0
# WHY: imageio 2.28+ 에서 GIF duration 단위가 '초'→'밀리초'로 바뀌어 이전엔 fps가
#      무시되고 느리게(≈10fps) 재생됐다. 현재는 ms로 보정돼 GIF_FPS가 실제 속도에 반영된다.
#      (GIF는 10ms 해상도라 fps가 근사될 수 있다. 원본 25fps→40ms는 정확.)
GIF_FPS = FPS   # 원본과 동일

# segments: 셀 9.5에서 정의(슬로우모션). 빈 튜플이면 등속(기존 동작 무회귀).
gif_config = VideoExportConfig(
    fmt="gif",
    fps=GIF_FPS,
    gap_policy=GapPolicy.BACKGROUND,  # occlusion 구간도 마지막 위치 유지(GIF 끊김 방지)
    segments=SEGMENTS,                # 구간별 슬로우/패스트(GIF는 per-frame duration)
)
usecase.export(
    frames=frames,
    boxes=boxes,
    target=(GIF_PATH, gif_config),
    result=track_result,
)
print(f"{GIF_PATH} 생성 완료 (GIF_FPS={GIF_FPS:.1f}, 구간 {len(SEGMENTS)}개)")

# MP4 — 원본 FPS 유지, FREEZE 정책(occlusion 구간 마지막 유효 프레임 반복)
# 슬로우모션은 MP4에서 프레임 복제/드롭(CFR)으로 표현된다.
mp4_config = VideoExportConfig(
    fmt="mp4",
    fps=float(FPS),
    gap_policy=GapPolicy.FREEZE,
    segments=SEGMENTS,                # 구간별 슬로우/패스트(MP4는 프레임 복제/드롭)
)
usecase.export(
    frames=frames,
    boxes=boxes,
    target=(MP4_PATH, mp4_config),
    result=track_result,
)
print(f"{MP4_PATH} 생성 완료 (fps={FPS:.1f}, 구간 {len(SEGMENTS)}개)")

In [ ]:
# ───────────────────────────────────────────────
# 셀 11: 추적 오버레이 영상 생성 (눈으로 추적 품질 확인)
# ───────────────────────────────────────────────
# TrackResult.masks로 마스크 오버레이 영상을 만들어 추적이 올바른지 시각 확인한다.
# 기존 PoC 노트북 셀 8 패턴을 그대로 계승한다.
import imageio

OVERLAY_PATH = "track_overlay.mp4"

def _draw_overlay(frame: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """단일 프레임에 마스크 오버레이를 그린다.
    
    mask가 None이거나 비어있으면 원본 그대로 반환한다.
    WHY: TrackResult.masks에는 occlusion 프레임도 빈 배열로 존재하므로
         sum()으로 유효 여부를 판단한다.
    """
    out = frame.copy()
    if mask is None or not mask.any():
        return out
    # 마스크 영역을 빨간색으로 반투명 오버레이
    out[mask] = (
        0.45 * out[mask] + 0.55 * np.array([255, 40, 40])
    ).astype(np.uint8)
    # 마스크 bbox 테두리
    ys, xs = np.where(mask)
    if len(xs) > 0:
        cv2.rectangle(
            out,
            (int(xs.min()), int(ys.min())),
            (int(xs.max()), int(ys.max())),
            (255, 40, 40), 2,
        )
    return out

overlay_frames = [
    _draw_overlay(frames[i], track_result.masks[i])
    for i in range(len(frames))
]

imageio.mimsave(OVERLAY_PATH, overlay_frames, fps=FPS)
print(f"{OVERLAY_PATH} 생성 완료")

In [ ]:
# ───────────────────────────────────────────────
# 셀 12: 결과물 다운로드
# ───────────────────────────────────────────────
from google.colab import files
import os

# 각 파일이 실제 생성됐는지 확인 후 다운로드
# WHY: export 오류로 파일이 없으면 files.download가 조용히 실패하므로 사전 검증
for path in [GIF_PATH, MP4_PATH, OVERLAY_PATH]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"다운로드: {path} ({size_kb:.1f} KB)")
        files.download(path)
    else:
        print(f"오류: {path} 파일이 없습니다. 이전 셀에서 오류가 있었는지 확인하세요.")

## 결과 해석

### 크롭 정합성 확인 (핵심)

`clip_crop.gif` / `clip_crop.mp4`를 열어 다음을 확인한다.

- **정상**: 크롭 프레임이 피사체를 **따라가며 중심에 유지**된다.
- **비정상**: 좌상단 배경만 보인다 → `compute_boxes` 또는 centroid 추출 오류.

기존 PoC 노트북(`easy_capture_gpu_poc.ipynb`) 셀 9는 `ffmpeg crop=320:320:0:0`(좌상단 고정)을 사용해 항상 배경만 잘렸다.  
이 노트북은 `VideoCaptureUseCase.compute_boxes()`가 `TrackResult.centroids`를 기반으로 박스를 계산하므로 피사체를 따라간다.

### 마스크 해상도 정합 확인

`track_overlay.mp4`에서 마스크가 피사체 영역에 정확히 겹치는지 확인한다.  
마스크가 피사체와 어긋나거나 프레임 전체를 덮으면 `Sam2VideoBackend._extract_mask()`의 `post_process_masks(original_sizes=...)` 인자가 잘못된 것이다.

### AC 기준

| 항목 | 기준 | 확인 방법 |
|------|------|----------|
| **AC-01** 추적 유지율 | ≥ 80% | 셀 8 콘솔 출력 |
| **AC-06** GPU 속도 | ≥ 10 fps | 셀 8 콘솔 출력 |
| 크롭 정합성 | 피사체 중심 크롭 | `clip_crop.gif` 육안 확인 |
| 마스크 해상도 | 피사체 영역 정확히 오버레이 | `track_overlay.mp4` 육안 확인 |

### 조정 포인트

- **추적 유지율 낮음**: `TARGET_IDX` 변경(더 선명한 후보 선택), `MAX_SECONDS` 단축.
- **크롭이 흔들림**: `smooth_window` 값 증가(예: 5 → 9).
- **후보 검출 없음**: `DETECT_PROMPT` 변경(예: `"person."` → `"singer."`), `GroundingDinoBackend(box_threshold=0.2)` 임계값 낮춤.
- **GPU OOM**: `SAM2_REPO`를 `"facebook/sam2.1-hiera-tiny"`로 유지하거나 `MAX_SECONDS` 단축.

결과 수치를 `poc/REPORT.md`의 미검증 항목에 채워 **Go/No-Go 확정**한다.